# Milestone 1 — Dataset Design
## AI Bill Anomaly Detection System

This notebook generates a **synthetic invoice dataset** (~10,000 rows) that mimics real
government/organizational billing data, with **deliberate anomalies injected** so we have
ground truth to validate our Isolation Forest model against later.

**Anomaly types injected:**
1. Duplicate invoices (~2%)
2. Abnormally high amounts vs department average (~3%)
3. Incorrect GST calculations (~2%)
4. Abnormally high-frequency vendors (a few vendors submitting way more invoices than normal)

Each injected anomaly gets a `True_Anomaly` label (1 = anomaly, 0 = normal) — this is **not**
used by the model (it's unsupervised), but we'll use it later in Milestone 6 to evaluate
precision/recall of Isolation Forest's predictions.

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

N_ROWS = 10000
print(f"Target dataset size: {N_ROWS} rows")

Target dataset size: 10000 rows


## Step 1: Define reference data (vendors, departments, categories)

In [2]:
departments = [
    "Public Works", "Health", "Education", "Transport",
    "IT & Electronics", "Sanitation", "Administration", "Water Supply"
]

categories = [
    "Stationery", "Electronics", "Furniture", "Medical Supplies",
    "Construction Material", "Vehicle Maintenance", "Software License",
    "Cleaning Supplies", "Fuel", "Office Equipment"
]

# 40 normal vendors + a few "high frequency" vendors added later
vendors = [f"Vendor_{i:03d}" for i in range(1, 41)]

# Typical amount ranges per department (mean, std) — used to generate realistic normal spend
dept_amount_profile = {
    dept: (np.random.randint(5000, 25000), np.random.randint(1000, 5000))
    for dept in departments
}

gst_rates = [5, 12, 18, 28]  # standard GST slabs (India)

print("Departments:", departments)
print("Sample amount profile:", dict(list(dept_amount_profile.items())[:3]))

Departments: ['Public Works', 'Health', 'Education', 'Transport', 'IT & Electronics', 'Sanitation', 'Administration', 'Water Supply']
Sample amount profile: {'Public Works': (20795, 1860), 'Health': (10390, 2130), 'Education': (16964, 4092)}


## Step 2: Generate normal (non-anomalous) invoice records

In [3]:
def random_date(start, end):
    delta = end - start
    random_days = random.randint(0, delta.days)
    return start + timedelta(days=random_days)

start_date = datetime(2024, 1, 1)
end_date = datetime(2025, 12, 31)

records = []

for i in range(N_ROWS):
    dept = random.choice(departments)
    vendor = random.choice(vendors)
    category = random.choice(categories)
    invoice_date = random_date(start_date, end_date)
    quantity = np.random.randint(1, 50)

    mean_amt, std_amt = dept_amount_profile[dept]
    amount = max(500, np.random.normal(mean_amt, std_amt))

    gst_pct = random.choice(gst_rates)

    records.append({
        "Invoice_ID": f"INV{100000 + i}",
        "Vendor_Name": vendor,
        "Department": dept,
        "Invoice_Date": invoice_date,
        "Amount": round(amount, 2),
        "GST": gst_pct,
        "Quantity": quantity,
        "Category": category,
        "True_Anomaly": 0  # normal by default, may be overwritten below
    })

df = pd.DataFrame(records)
df.head()

,Invoice_ID,Vendor_Name,Department,Invoice_Date,Amount,GST,Quantity,Category,True_Anomaly
0,INV100000,Vendor_002,Health,2024-09-07,10560.04,12,22,Construction Material,0
1,INV100001,Vendor_007,Education,2024-03-30,16311.26,28,30,Fuel,0
2,INV100002,Vendor_002,Public Works,2024-08-11,23515.02,12,38,Electronics,0
3,INV100003,Vendor_036,Public Works,2025-10-27,23657.01,28,44,Medical Supplies,0
4,INV100004,Vendor_029,Transport,2024-10-11,12461.91,5,25,Office Equipment,0


## Step 3: Inject anomalies

We inject four anomaly types into randomly selected rows. Each injected row is flagged
`True_Anomaly = 1`.

In [4]:
# --- Anomaly 1: Duplicate invoices (~2%) ---
n_duplicates = int(N_ROWS * 0.02)
dup_source_idx = np.random.choice(df.index, n_duplicates, replace=False)
dup_rows = df.loc[dup_source_idx].copy()
dup_rows["Invoice_ID"] = [f"INV{200000 + i}" for i in range(n_duplicates)]
dup_rows["True_Anomaly"] = 1
# keep same Vendor, Amount, Date as the original -> genuine duplicate pattern

print(f"Injected {n_duplicates} duplicate invoices")
dup_rows.head(3)

Injected 200 duplicate invoices


,Invoice_ID,Vendor_Name,Department,Invoice_Date,Amount,GST,Quantity,Category,True_Anomaly
6552,INV200000,Vendor_028,Public Works,2024-05-25,23021.75,28,4,Electronics,1
953,INV200001,Vendor_019,Transport,2025-02-12,8683.72,18,29,Electronics,1
606,INV200002,Vendor_008,Administration,2025-05-26,13815.75,5,3,Medical Supplies,1


In [5]:
# --- Anomaly 2: Abnormally high amounts (~3%) ---
n_high_amount = int(N_ROWS * 0.03)
high_idx = np.random.choice(df.index, n_high_amount, replace=False)

for idx in high_idx:
    dept = df.loc[idx, "Department"]
    mean_amt, _ = dept_amount_profile[dept]
    inflated = mean_amt * np.random.uniform(6, 12)  # 6x-12x the department average
    df.loc[idx, "Amount"] = round(inflated, 2)
    df.loc[idx, "True_Anomaly"] = 1

print(f"Injected {n_high_amount} abnormally high-amount invoices")

Injected 300 abnormally high-amount invoices


In [6]:
# --- Anomaly 3: Incorrect GST calculation (~2%) ---
# We simulate this by assigning a GST value outside the valid slabs (e.g. 40%, 55%, 2%)
n_wrong_gst = int(N_ROWS * 0.02)
gst_idx = np.random.choice(df.index, n_wrong_gst, replace=False)
invalid_gst_values = [1, 2, 3, 35, 40, 45, 55]

for idx in gst_idx:
    df.loc[idx, "GST"] = random.choice(invalid_gst_values)
    df.loc[idx, "True_Anomaly"] = 1

print(f"Injected {n_wrong_gst} incorrect GST invoices")

Injected 200 incorrect GST invoices


In [7]:
# --- Anomaly 4: Abnormally high-frequency vendors ---
# Introduce 2 "shell-like" vendors that submit far more invoices than normal vendors
suspicious_vendors = ["Vendor_999", "Vendor_998"]
n_suspicious_rows = int(N_ROWS * 0.015)
susp_idx = np.random.choice(df.index, n_suspicious_rows, replace=False)

for idx in susp_idx:
    df.loc[idx, "Vendor_Name"] = random.choice(suspicious_vendors)
    df.loc[idx, "True_Anomaly"] = 1

print(f"Reassigned {n_suspicious_rows} invoices to high-frequency suspicious vendors")

Reassigned 150 invoices to high-frequency suspicious vendors


In [8]:
# Combine everything: original df (now with amount/gst/vendor anomalies applied in-place)
# + the newly created duplicate rows
df_final = pd.concat([df, dup_rows], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle rows

print("Final dataset shape:", df_final.shape)
print("\nAnomaly distribution:")
print(df_final["True_Anomaly"].value_counts())
print(f"\nAnomaly rate: {df_final['True_Anomaly'].mean()*100:.2f}%")

Final dataset shape: (10200, 9)

Anomaly distribution:
True_Anomaly
0    9358
1     842
Name: count, dtype: int64

Anomaly rate: 8.25%


## Step 4: Sanity checks before saving

In [9]:
print("Missing values:\n", df_final.isnull().sum())
print("\nDtypes:\n", df_final.dtypes)
print("\nSample rows:")
df_final.sample(5)

Missing values:
 Invoice_ID      0
Vendor_Name     0
Department      0
Invoice_Date    0
Amount          0
GST             0
Quantity        0
Category        0
True_Anomaly    0
dtype: int64

Dtypes:
 Invoice_ID                 str
Vendor_Name                str
Department                 str
Invoice_Date    datetime64[us]
Amount                 float64
GST                      int64
Quantity                 int64
Category                   str
True_Anomaly             int64
dtype: object

Sample rows:


,Invoice_ID,Vendor_Name,Department,Invoice_Date,Amount,GST,Quantity,Category,True_Anomaly
5224,INV109693,Vendor_013,Transport,2024-11-19,11080.78,12,49,Electronics,0
7481,INV102326,Vendor_009,Administration,2025-12-19,12724.71,28,17,Software License,0
5374,INV100975,Vendor_032,Sanitation,2024-09-13,17647.02,5,24,Construction Material,0
5352,INV105819,Vendor_035,Water Supply,2025-12-14,21920.78,5,45,Vehicle Maintenance,0
7728,INV106285,Vendor_010,Education,2024-06-07,26083.07,28,21,Vehicle Maintenance,0


In [10]:
import os
os.makedirs("../data/raw", exist_ok=True)
output_path = "../data/raw/invoices.csv"
df_final.to_csv(output_path, index=False)
print(f"Saved dataset to {output_path}")
print(f"Total rows: {len(df_final)}")

Saved dataset to ../data/raw/invoices.csv
Total rows: 10200


## Notes for Milestone 2 (Data Cleaning)

- `True_Anomaly` is our **ground truth label** — keep it in `data/raw/`, but **drop it before
  feeding features into Isolation Forest** (it's unsupervised). Use it only at evaluation time
  in Milestone 6.
- Don't "clean away" the injected anomalies (e.g. don't cap outlier amounts) — that would
  destroy the signal you're trying to detect. Cleaning should target genuine noise (nulls,
  duplicate *unintentional* records, bad dtypes), not your injected anomalies.
- Next notebook: `02_data_cleaning.ipynb`